# IT Incident Response Agentic AI — Python 3.11.9 Compatible

This notebook demonstrates **Agentic AI** using multiple specialized LLM roles.

It intentionally avoids:

```python
from langchain.agents import create_agent
```

and avoids direct LangGraph imports.

This prevents the LangGraph checkpoint compatibility error in your current environment.

The LLM integration remains:

```python
from langchain_openai import ChatOpenAI
```

## Workflow

```text
Incident
   ↓
Triage Agent
   ↓
Diagnostic Agent
   ↓
Remediation Agent
   ↓
Human Approval Gate
   ↓
Communication Agent
```

## Step 1 — Check environment

In [ ]:
import sys
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version)

for pkg in [
    "langchain-core",
    "langchain-openai",
    "openai",
    "pandas",
    "python-dotenv",
]:
    try:
        print(f"{pkg}: {version(pkg)}")
    except PackageNotFoundError:
        print(f"{pkg}: NOT INSTALLED")

## Step 2 — Imports and API key

In [ ]:
import os
import json
import pandas as pd

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not found in the .env file.")

print("Imports loaded successfully.")

## Step 3 — Load incident data

Keep `agentic_it_incidents.csv` in the same folder.

In [ ]:
incidents = pd.read_csv("agentic_it_incidents.csv")

display(incidents)

## Step 4 — Investigation tools

The workflow receives read-only tools.

This keeps the demo safe: agents can investigate and recommend, but do not directly restart or modify production.

In [ ]:
@tool
def get_incident(incident_id: str) -> str:
    """Retrieve one incident by incident ID."""
    incident_id = str(incident_id).strip().upper()

    row = incidents[
        incidents["incident_id"].str.upper() == incident_id
    ]

    if row.empty:
        return f"Incident {incident_id} was not found."

    return row.iloc[0].to_json()


@tool
def get_service_runbook(service: str) -> str:
    """Return the operational runbook for a service."""

    runbooks = {
        "auth-service": (
            "Check token provider health; inspect authentication errors; "
            "verify dependency latency and recent timeout changes."
        ),
        "orders-db": (
            "Check CPU and active sessions; inspect top SQL; "
            "verify indexes before proposing changes."
        ),
        "payments-api": (
            "Inspect downstream dependencies, latency percentiles, "
            "database pool saturation, and recent configuration changes."
        ),
        "checkout-service": (
            "Inspect pod memory, OOMKilled events, memory requests/limits, "
            "and recent deployments."
        ),
    }

    return runbooks.get(
        service,
        "Collect logs, metrics, traces, deployment history, and dependency health."
    )


@tool
def get_recent_change(service: str) -> str:
    """Return a simulated recent deployment or configuration change."""

    changes = {
        "auth-service": (
            "Version 3.8.1 deployed 35 minutes before the incident; "
            "token-validation timeout changed from 2s to 1s."
        ),
        "orders-db": (
            "A new reporting query was released this morning."
        ),
        "payments-api": (
            "Connection pool max size was reduced from 120 to 80."
        ),
        "checkout-service": (
            "Version 5.4.0 was deployed with a new image-processing dependency."
        ),
    }

    return changes.get(
        service,
        "No significant recent change found."
    )

## Step 5 — Shared LLM

In [ ]:
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
)

## Step 6 — Generic tool-enabled specialist runner

We reuse one small tool-calling loop for specialist agents that need tools.

In [ ]:
investigation_tools = [
    get_incident,
    get_service_runbook,
    get_recent_change,
]

investigation_tool_map = {
    t.name: t for t in investigation_tools
}


def run_tool_agent(
    system_prompt: str,
    user_prompt: str,
    tools: list,
    max_iterations: int = 5,
) -> dict:

    tool_map = {t.name: t for t in tools}
    model = llm.bind_tools(tools)

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ]

    trace = []

    for _ in range(max_iterations):
        ai_message = model.invoke(messages)
        messages.append(ai_message)

        if not ai_message.tool_calls:
            return {
                "answer": ai_message.content,
                "trace": trace,
            }

        for tool_call in ai_message.tool_calls:
            name = tool_call["name"]
            args = tool_call["args"]
            call_id = tool_call["id"]

            if name in tool_map:
                output = tool_map[name].invoke(args)
            else:
                output = f"Unknown tool: {name}"

            trace.append({
                "tool": name,
                "arguments": args,
                "output": str(output),
            })

            messages.append(
                ToolMessage(
                    content=str(output),
                    tool_call_id=call_id,
                )
            )

    return {
        "answer": "Maximum iterations reached.",
        "trace": trace,
    }


def run_text_agent(system_prompt: str, user_prompt: str) -> str:
    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt),
    ])
    return response.content

## Step 7 — Define specialist roles

In [ ]:
TRIAGE_PROMPT = """
You are the Triage Agent.

Retrieve the incident and provide:
1. Severity assessment.
2. Business impact.
3. Immediate investigation priority.
4. Escalation recommendation.

Do not invent facts and do not claim to execute production changes.
"""


DIAGNOSTIC_PROMPT = """
You are the Diagnostic Agent.

Use incident data, runbooks, and recent changes to provide:
1. Root-cause hypotheses.
2. Evidence for each hypothesis.
3. Additional checks.
4. Most likely cause.

Do not execute remediation.
"""


REMEDIATION_PROMPT = """
You are the Remediation Planning Agent.

Create a safe remediation plan from the supplied triage and diagnosis.

Rules:
- Start with low-risk/reversible actions.
- Include validation checks.
- Mark restart, rollback, scale, configuration, database, deployment,
  or traffic changes as HUMAN APPROVAL REQUIRED.
- Never claim that an action was executed.
"""


COMMUNICATION_PROMPT = """
You are the Incident Communication Agent.

Create a concise stakeholder update containing:
- Incident summary.
- Customer/business impact.
- Current investigation status.
- Current remediation status.

Do not invent an ETA.
"""

## Step 8 — Build the multi-agent workflow

Outputs from earlier agents are passed to later agents.

That state handoff is a core Agentic AI concept.

In [ ]:
def run_incident_workflow(
    incident_id: str,
    human_approved: bool = False,
) -> dict:

    incident_id = str(incident_id).strip().upper()
    workflow_trace = []

    # 1. TRIAGE
    triage = run_tool_agent(
        TRIAGE_PROMPT,
        f"Triage incident {incident_id}.",
        [get_incident],
    )
    workflow_trace.append({
        "agent": "Triage Agent",
        "tools": triage["trace"],
    })

    # 2. DIAGNOSIS
    diagnosis = run_tool_agent(
        DIAGNOSTIC_PROMPT,
        (
            f"Investigate incident {incident_id}.\n\n"
            f"TRIAGE NOTE:\n{triage['answer']}"
        ),
        investigation_tools,
    )
    workflow_trace.append({
        "agent": "Diagnostic Agent",
        "tools": diagnosis["trace"],
    })

    # 3. REMEDIATION PLANNING
    remediation = run_text_agent(
        REMEDIATION_PROMPT,
        (
            f"INCIDENT: {incident_id}\n\n"
            f"TRIAGE:\n{triage['answer']}\n\n"
            f"DIAGNOSIS:\n{diagnosis['answer']}"
        ),
    )
    workflow_trace.append({
        "agent": "Remediation Agent",
        "tools": [],
    })

    # 4. HUMAN APPROVAL GATE
    if human_approved:
        action_status = (
            "Human approval received. The approved plan may be passed "
            "to an authorized operational system. "
            "This notebook does not execute production changes."
        )
    else:
        action_status = (
            "No production-changing action executed. "
            "Human approval is required."
        )

    # 5. COMMUNICATION
    communication = run_text_agent(
        COMMUNICATION_PROMPT,
        (
            f"INCIDENT: {incident_id}\n\n"
            f"TRIAGE:\n{triage['answer']}\n\n"
            f"DIAGNOSIS:\n{diagnosis['answer']}\n\n"
            f"REMEDIATION PLAN:\n{remediation}\n\n"
            f"ACTION STATUS:\n{action_status}"
        ),
    )

    workflow_trace.append({
        "agent": "Communication Agent",
        "tools": [],
    })

    return {
        "incident_id": incident_id,
        "triage": triage["answer"],
        "diagnosis": diagnosis["answer"],
        "remediation_plan": remediation,
        "action_status": action_status,
        "communication": communication,
        "trace": workflow_trace,
    }

## Step 9 — Run the Agentic AI workflow

In [ ]:
state = run_incident_workflow(
    incident_id="INC002",
    human_approved=False,
)

print("TRIAGE\n")
print(state["triage"])

print("\nDIAGNOSIS\n")
print(state["diagnosis"])

print("\nREMEDIATION\n")
print(state["remediation_plan"])

print("\nACTION STATUS\n")
print(state["action_status"])

print("\nCOMMUNICATION\n")
print(state["communication"])

## Step 10 — Observe agent and tool handoffs

In [ ]:
trace_rows = []

for item in state["trace"]:
    if item["tools"]:
        for tool_event in item["tools"]:
            trace_rows.append({
                "agent": item["agent"],
                "tool": tool_event["tool"],
                "arguments": tool_event["arguments"],
                "output": tool_event["output"],
            })
    else:
        trace_rows.append({
            "agent": item["agent"],
            "tool": "No tool",
            "arguments": "",
            "output": "",
        })

display(pd.DataFrame(trace_rows))

## Step 11 — Human approval example

In [ ]:
approved_state = run_incident_workflow(
    incident_id="INC002",
    human_approved=True,
)

print(approved_state["action_status"])

## Why this is Agentic AI

The workflow has multiple specialized roles:

```text
Triage → Diagnosis → Remediation → Approval → Communication
```

The agents collaborate by passing context/state to the next stage.

This version remains fully based on `langchain_openai.ChatOpenAI`, but does not depend on `langchain.agents.create_agent` or LangGraph internals.